In [53]:
pip install langgraph langchain-core langchain-ollama gradio notebook ipykernel python-dotenv requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [54]:
# Cell 1 — Imports & sanity check
import langgraph
import langchain_core
import requests
import gradio
from importlib.metadata import version

print(f"langgraph:      {version('langgraph')}")
print(f"langchain_core: {version('langchain-core')}")
print(f"gradio:         {gradio.__version__}")

# Verify Ollama is reachable
response = requests.get("http://localhost:11434/api/tags")
if response.status_code == 200:
    models = [m["name"] for m in response.json()["models"]]
    print(f"\nOllama running ✓")
    print(f"Available models: {models}")
else:
    print("Ollama not reachable — run 'ollama serve' in terminal")

langgraph:      1.2.2
langchain_core: 1.4.0
gradio:         6.15.2

Ollama running ✓
Available models: ['llama3.1:8b']


In [55]:
# Cell 2 — Agent state definition
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    # The research topic the user wants investigated
    topic: str
    
    # Full message history — add_messages appends instead of overwriting
    messages: Annotated[list, add_messages]
    
    # Search results collected by the agent
    search_results: list[str]
    
    # Number of steps taken — for termination
    steps_taken: int
    
    # Maximum steps allowed — resource-based termination
    max_steps: int
    
    # Current status of the agent
    status: str  # "running", "done", "error"
    
    # The final research report
    report: str

print("Agent state defined.")
print("\nState fields:")
for field, annotation in AgentState.__annotations__.items():
    print(f"  {field}: {annotation}")

Agent state defined.

State fields:
  topic: <class 'str'>
  messages: typing.Annotated[list, <function _add_messages_wrapper.<locals>._add_messages at 0x11672f690>]
  search_results: list[str]
  steps_taken: <class 'int'>
  max_steps: <class 'int'>
  status: <class 'str'>
  report: <class 'str'>


In [56]:
# Cell 3 — Ollama direct API client
import requests
import json

def call_ollama(prompt: str, system: str = "") -> str:
    """Call Ollama directly via REST API."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "model": "llama3.1:8b",
        "messages": messages,
        "stream": False
    }
    
    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=60
    )
    
    result = response.json()
    return result["message"]["content"]

def search_web(query: str) -> str:
    """Search DuckDuckGo and return results."""
    try:
        from bs4 import BeautifulSoup
        headers = {"User-Agent": "Mozilla/5.0"}
        url = f"https://duckduckgo.com/html/?q={query.replace(' ', '+')}"
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        
        results = []
        for result in soup.find_all("div", class_="result__body")[:5]:
            text = result.get_text(strip=True)
            if text:
                results.append(text[:300])
        
        return "\n\n".join(results) if results else "No results found."
    except Exception as e:
        return f"Search failed: {str(e)}"

# Test both
print("Testing Ollama connection...")
response = call_ollama("What is LoRA in machine learning? One sentence.")
print(f"LLM response: {response}")

print("\nTesting web search...")
results = search_web("LoRA fine-tuning NLP")
print(f"Search results preview: {results[:200]}")

Testing Ollama connection...
LLM response: LoRA (Learning to Linearize Activations) is a technique for reducing the computational cost of large neural networks by adapting pre-trained weights, but allowing the adaptation to be stored efficiently and easily transferred between models, making it particularly useful for on-device machine learning applications.

Testing web search...
Search results preview: Fine-Tuning using LoRA and QLoRA - GeeksforGeekswww.geeksforgeeks.org/deep-learning/fine-tuning-using-lora-and-qlora/Finetuningupdates all the parameters of a pre-trained model to adapt it for a speci


In [57]:
# Cell 4 — Agent nodes using direct Ollama API
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    topic: str
    search_results: list[str]
    steps_taken: int
    max_steps: int
    status: str
    report: str
    next_query: str

def planner_node(state: AgentState) -> AgentState:
    print(f"\n[Planner] Step {state['steps_taken'] + 1}/{state['max_steps']}")
    
    results_text = "\n\n".join(state["search_results"]) if state["search_results"] else "None yet"
    
    system = f"""You are a research agent. Your goal is to research: "{state['topic']}"

Current search results:
{results_text}

You must respond in ONE of these two formats only:

FORMAT 1 - If you need more information:
SEARCH: <your search query here>

FORMAT 2 - If you have enough information (at least 2 good results):
RESEARCH_COMPLETE

Be decisive. Do not repeat searches."""

    response = call_ollama(
        prompt=f"Research topic: {state['topic']}. What should I do next?",
        system=system
    )
    
    print(f"[Planner] Decision: {response[:100]}")
    
    if "RESEARCH_COMPLETE" in response:
        return {
            "steps_taken": state["steps_taken"] + 1,
            "status": "done",
            "next_query": ""
        }
    
    # Extract search query
    query = state["topic"]  # default
    if "SEARCH:" in response:
        query = response.split("SEARCH:")[-1].strip().split("\n")[0].strip()
    
    return {
        "steps_taken": state["steps_taken"] + 1,
        "status": "running",
        "next_query": query
    }

def search_node(state: AgentState) -> AgentState:
    query = state["next_query"] or state["topic"]
    print(f"[Search] Searching: '{query}'")
    
    result = search_web(query)
    print(f"[Search] Got {len(result)} chars")
    
    return {
        "search_results": state["search_results"] + [f"Query: {query}\n{result[:400]}"]
    }

def synthesiser_node(state: AgentState) -> AgentState:
    print("\n[Synthesiser] Writing final report...")
    
    results_text = "\n\n".join(state["search_results"])
    
    report = call_ollama(
        prompt=f"Write a structured research report about: {state['topic']}\n\nBased on these results:\n{results_text}",
        system="""You are a research report writer. Write a clear report with:
1. Brief overview (2-3 sentences)
2. Key findings (3-5 bullet points)
3. Conclusion (1-2 sentences)

Be concise and informative."""
    )
    
    print(f"[Synthesiser] Report length: {len(report)} chars")
    
    return {
        "report": report,
        "status": "done"
    }

print("Nodes defined ✓")

Nodes defined ✓


In [58]:
# Cell 5 — Build the graph
from langgraph.graph import StateGraph, END

graph_builder = StateGraph(AgentState)

graph_builder.add_node("planner", planner_node)
graph_builder.add_node("search", search_node)
graph_builder.add_node("synthesiser", synthesiser_node)

graph_builder.set_entry_point("planner")

def route_after_planner(state: AgentState) -> str:
    if state["steps_taken"] >= state["max_steps"]:
        print("[Router] Max steps reached → synthesise")
        return "synthesiser"
    if state["status"] == "done":
        print("[Router] Research complete → synthesise")
        return "synthesiser"
    print("[Router] Need more info → search")
    return "search"

def route_after_search(state: AgentState) -> str:
    return "planner"

graph_builder.add_conditional_edges("planner", route_after_planner)
graph_builder.add_conditional_edges("search", route_after_search)
graph_builder.add_edge("synthesiser", END)

agent = graph_builder.compile()

print("Agent compiled ✓")
print("\nGraph:")
print("  START → planner")
print("  planner → [search | synthesiser] (conditional)")
print("  search → planner")
print("  synthesiser → END")

Agent compiled ✓

Graph:
  START → planner
  planner → [search | synthesiser] (conditional)
  search → planner
  synthesiser → END


In [59]:
# Cell 6 — Run the agent
def run_agent(topic: str, max_steps: int = 5):
    print(f"Starting research agent...")
    print(f"Topic: '{topic}'")
    print(f"Max steps: {max_steps}")
    print("=" * 50)
    
    initial_state = {
        "topic": topic,
        "search_results": [],
        "steps_taken": 0,
        "max_steps": max_steps,
        "status": "running",
        "report": "",
        "next_query": ""
    }
    
    final_state = agent.invoke(initial_state)
    
    print("\n" + "=" * 50)
    print("FINAL REPORT")
    print("=" * 50)
    print(final_state["report"])
    print(f"\nSteps taken: {final_state['steps_taken']}")
    print(f"Search results collected: {len(final_state['search_results'])}")
    
    return final_state

result = run_agent("LoRA fine-tuning for NLP models", max_steps=4)

Starting research agent...
Topic: 'LoRA fine-tuning for NLP models'
Max steps: 4

[Planner] Step 1/4
[Planner] Decision: SEARCH: "LoRA fine-tuning techniques for natural language processing" OR "applying LoRA adapters to 
[Router] Need more info → search
[Search] Searching: '"LoRA fine-tuning techniques for natural language processing" OR "applying LoRA adapters to NLP models"'
[Search] Got 199 chars

[Planner] Step 2/4
[Planner] Decision: FORMAT 1 - SEARCH:
"LoRA adapters for NLP model fine-tuning" OR "applying weight sharing with LoRA t
[Router] Need more info → search
[Search] Searching: '"LoRA adapters for NLP model fine-tuning" OR "applying weight sharing with LoRA to NLP tasks"'
[Search] Got 189 chars

[Planner] Step 3/4
[Planner] Decision: FORMAT 1 - Since we haven't found relevant results yet, I'll try to provide a more specific search q
[Router] Need more info → search
[Search] Searching: '"LoRA adapters" OR "weight sharing with LoRA" AND ("NLP model adaptation" OR "fine-tunin

In [60]:
# Cell 7 — Run the agent
from langchain_core.messages import HumanMessage

def run_agent(topic: str, max_steps: int = 5):
    print(f"Starting research agent...")
    print(f"Topic: '{topic}'")
    print(f"Max steps: {max_steps}")
    print("=" * 50)
    
    # Initial state
    initial_state = {
        "topic": topic,
        "messages": [HumanMessage(content=f"Research this topic thoroughly: {topic}")],
        "search_results": [],
        "steps_taken": 0,
        "max_steps": max_steps,
        "status": "running",
        "report": ""
    }
    
    # Run the agent
    final_state = agent.invoke(initial_state)
    
    print("\n" + "=" * 50)
    print("FINAL REPORT")
    print("=" * 50)
    print(final_state["report"])
    print(f"\nSteps taken: {final_state['steps_taken']}")
    print(f"Status: {final_state['status']}")
    
    return final_state

# Test with a topic relevant to what you've learned
result = run_agent("LoRA fine-tuning for NLP models", max_steps=4)

Starting research agent...
Topic: 'LoRA fine-tuning for NLP models'
Max steps: 4

[Planner] Step 1/4
[Planner] Decision: SEARCH: (LoRA) Low-Rank Adaptation for Neural Networks + fine tuning NLP + recent research papers or
[Router] Need more info → search
[Search] Searching: '(LoRA) Low-Rank Adaptation for Neural Networks + fine tuning NLP + recent research papers or publications on the topic'
[Search] Got 1491 chars

[Planner] Step 2/4
[Planner] Decision: FORMAT 1 - If you need more information:
SEARCH: "LoRA fine-tuning for NLP models" recent research p
[Router] Need more info → search
[Search] Searching: '"LoRA fine-tuning for NLP models" recent research papers or publications on the topic, including comparisons with other parameter-efficient fine-tuning methods like PEFT and La-LoRA.'
[Search] Got 278 chars

[Planner] Step 3/4
[Planner] Decision: FORMAT 1 - SEARCH:

LoRA (Low-Rank Adaptation) fine-tuning for NLP models: comparisons with PEFT and
[Router] Need more info → search
[Sea

In [61]:
# Debug cell — test synthesiser LLM directly
from langchain_core.messages import SystemMessage

test_prompt = """Write a 3 sentence summary about LoRA fine-tuning for NLP models."""

response = llm.invoke([SystemMessage(content=test_prompt)])
print(f"Response length: {len(response.content)}")
print(f"Response: {response.content}")

Response length: 0
Response: 


In [62]:
# Fix — call Ollama directly via REST API
import requests
import json

def call_ollama(prompt: str, system: str = "") -> str:
    payload = {
        "model": "llama3.1:8b",
        "messages": [
            {"role": "system", "content": system} if system else None,
            {"role": "user", "content": prompt}
        ],
        "stream": False
    }
    # Remove None values
    payload["messages"] = [m for m in payload["messages"] if m]
    
    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=60
    )
    
    result = response.json()
    return result["message"]["content"]

# Test it
test = call_ollama(
    prompt="Write a 3 sentence summary about LoRA fine-tuning for NLP models.",
    system="You are a helpful research assistant."
)
print(f"Response length: {len(test)}")
print(f"Response: {test}")

Response length: 594
Response: LoRA (Learning to Adapt Linear Representations) is a technique that allows for efficient fine-tuning of pre-trained language models by learning a small set of adapter layers on top of the original model weights, rather than retraining the entire model from scratch. This approach enables fast and robust adaptation of large-scale NLP models to specific downstream tasks with minimal computational overhead. By using LoRA fine-tuning, researchers can leverage the strengths of pre-trained language models while also accommodating task-specific requirements in a computationally efficient manner.
